# The Supply Line: Solutions Notebook

This notebook contains complete solutions for both TODOs.

In [ ]:
# Setup
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists('coding-exercises'):
        !cd coding-exercises && git pull
    else:
        !git clone -b week3 https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir('coding-exercises/the_supply_line')

sys.path.insert(0, '.')

!pip install google-genai --quiet
print('Setup complete!')

In [ ]:
from supply_line import *
from supply_line.game import SupplyWorld, AgentState
from supply_line.data import OrderCategory, OrderPriority, OrderStatus
from supply_line.agent import TOOLS_DESCRIPTION, parse_action
print('The Supply Line loaded!')

agent, world, tools = create_game()
print()
print(world.queue_summary())

---
## Play the Game Yourself!

In [ ]:
from supply_line.interactive import play_interactive

game = play_interactive()

---
## Solution: TODO 1 — Rule-Based Think Function

Strategy: process orders as a state machine. Each order goes through phases:
1. Read it
2. Lookup KB if needed
3. Apply the correct action (place_order, submit_documents, reject_shipment, etc.)
4. Handle multi-step orders (quality: reject + claim + reorder)
5. Escalate if required (pricing disputes to procurement, big orders to finance)
6. Notify client
7. Close the order

Key insight: the two boss orders (T-001 Aurora, T-020 Regulatory) have **dependencies** — they require
a prepared briefing, which requires resolving prerequisite orders first. The agent must do
**dependency resolution** — skip boss orders whose prerequisites aren't met, handle
prerequisites first, then come back. This is the Supply Line equivalent of BFS pathfinding.

In [ ]:
import re

def _pick_next_order(world):
    """Pick the next order from the queue, skipping those with unmet briefing prerequisites.
    
    This is the dependency resolution step — the Supply Line equivalent of BFS.
    """
    for candidate in world.queue:
        if candidate.status == OrderStatus.RESOLVED:
            continue
        if candidate.requires_briefing and not candidate.briefing_prepared:
            ready, _ = world.check_briefing_ready(candidate.id)
            if not ready:
                continue  # skip — prerequisites not resolved yet
        return candidate
    # All remaining have unmet prerequisites — pick first anyway
    return world.queue[0] if world.queue else None


def _extract_shipment_id(message: str) -> str:
    """Extract shipment ID (e.g. SH-1190) from the order message."""
    match = re.search(r'#(SH-\d+)', message)
    return match.group(1) if match else "SH-0000"


def think_rule_based(agent: AgentState, world: SupplyWorld, history: list[dict]) -> str:
    """Rule-based operations coordinator using a per-order state machine."""
    order = world.active_order

    # --- No active order? Open the next one from the queue ---
    if order is None or order.status == OrderStatus.RESOLVED:
        next_order = _pick_next_order(world)
        if next_order is None:
            return 'ACTION: check_orders()'
        return f'ACTION: read_order(order_id="{next_order.id}")'

    oid = order.id
    cat = order.category

    # --- Phase 1: Handle system alerts immediately ---
    if cat == OrderCategory.SYSTEM_ALERT:
        if not order.action_applied:
            return f'ACTION: acknowledge_alert(alert_id="{oid}")'
        # acknowledge_alert auto-resolves; move on
        next_order = _pick_next_order(world)
        if next_order:
            return f'ACTION: read_order(order_id="{next_order.id}")'
        return 'ACTION: check_orders()'

    # --- Phase 2: Boss fights — dependency-gated ---
    if cat == OrderCategory.LAUNCH:
        ready, missing = world.check_briefing_ready(oid)
        if ready and not order.briefing_prepared:
            return 'ACTION: prepare_launch_briefing()'
        if order.briefing_prepared and not order.action_applied:
            return f'ACTION: authorize_launch(order_id="{oid}")'
        if order.action_applied and not order.notification_sent:
            return 'ACTION: notify_client(template="launch_status")'
        if order.action_applied and order.notification_sent:
            return f'ACTION: close_order(order_id="{oid}")'
        # Prerequisites not met — switch away
        world.active_order = None
        return 'ACTION: check_orders()'

    if cat == OrderCategory.REGULATORY:
        ready, missing = world.check_briefing_ready(oid)
        if ready and not order.briefing_prepared:
            return 'ACTION: prepare_compliance_package()'
        if order.briefing_prepared and not order.action_applied:
            return f'ACTION: submit_compliance(order_id="{oid}")'
        if order.action_applied and not order.notification_sent:
            return 'ACTION: notify_client(template="compliance_status")'
        if order.action_applied and order.notification_sent:
            return f'ACTION: close_order(order_id="{oid}")'
        world.active_order = None
        return 'ACTION: check_orders()'

    # --- Phase 3: Lookup KB if needed ---
    if order.requires_lookup and not order.lookup_done:
        return f'ACTION: lookup_kb(query="{order.lookup_query}")'

    # --- Phase 4: Apply required action ---
    if order.requires_action and not order.action_applied:
        action = order.requires_action

        if action == "place_order":
            if cat == OrderCategory.RUSH:
                return 'ACTION: place_order(supplier="SteelWorks GmbH", sku="as_needed", quantity="500", expedited="true")'
            elif "coating" in order.message.lower() and "MechPro" not in order.message:
                return 'ACTION: place_order(supplier="PureChem Basel", sku="specialty_coating", quantity="200")'
            elif "MechPro" in order.message:
                return 'ACTION: place_order(supplier="MechPro Engineering", sku="spec_v2.1", quantity="1")'
            elif "ChemPure AG" in order.message and cat == OrderCategory.REORDER:
                return 'ACTION: place_order(supplier="ChemPure AG", sku="as_needed", quantity="50")'
            else:
                return 'ACTION: place_order(supplier="SteelWorks GmbH", sku="as_needed", quantity="50")'

        if action == "reject_shipment":
            sid = _extract_shipment_id(order.message)
            return f'ACTION: reject_shipment(shipment_id="{sid}", reason="QC failure contamination")'

        if action == "submit_documents":
            sid = _extract_shipment_id(order.message)
            return f'ACTION: submit_documents(shipment_id="{sid}", doc_type="certificate_of_origin")'

        if action == "update_timeline":
            return f'ACTION: update_timeline(order_id="{oid}", new_eta="5 business days")'

        if action == "file_claim":
            return 'ACTION: file_claim(supplier="SteelWorks GmbH", amount="5000")'

        if action == "compile_records":
            return 'ACTION: compile_records(period="30_days")'

        if action == "acknowledge_alert":
            return f'ACTION: acknowledge_alert(alert_id="{oid}")'

    # --- Phase 4b: Escalate if needed ---
    if order.requires_escalation and order.escalated_to != order.requires_escalation:
        return f'ACTION: escalate(department="{order.requires_escalation}", reason="as per procedure")'

    # --- Phase 4c: Multi-step quality handling (reject + claim + reorder) ---
    if cat == OrderCategory.QUALITY and order.action_applied:
        if not order.claim_filed:
            return 'ACTION: file_claim(supplier="ChemPure AG", amount="25000")'
        recent = [h for h in history if h["role"] == "action"]
        recent_actions = [a["content"] for a in recent[-10:]]
        if not any("place_order" in a for a in recent_actions):
            return 'ACTION: place_order(supplier="PureChem Basel", sku="specialty_coating", quantity="200")'

    # --- Phase 5: Notify client ---
    if order.correct_template and not order.notification_sent:
        return f'ACTION: notify_client(template="{order.correct_template}")'

    # --- Phase 6: Close ---
    return f'ACTION: close_order(order_id="{oid}")'

In [ ]:
result = play_rule_based(think_rule_based, max_turns=100, delay=0.05)
print(f"\nResult: {'Victory!' if result['won'] else 'Not quite...'} | "
      f"Resolved: {result['resolved']} | Quality: {result['quality']}% | "
      f"Tokens: {result['tokens']} | Turns: {result['turns']}")
print(f"Log: {result['log_file']}")

---
## Solution: TODO 2 — LLM-Powered Think Function

Architecture: three layers with clear separation of concerns.

1. **Autopilot** — handles ONLY mechanical tasks: queue management, system alerts, KB lookups, and boss fight state machines (dependency resolution). No decisions about actions, suppliers, or templates.
2. **Memory** — builds structured context for the LLM: order details, extracted entities (supplier, SKU, shipment ID), KB results, progress tracking, recent history. Shows the *required action type* and *extracted entities* but does NOT build the complete action call — the LLM assembles that.
3. **LLM** — makes the actual decisions: which action to take, which supplier, what parameters, which template, whether to escalate. The LLM reads the order message, extracted entities, and KB procedures to figure out what to do.
4. **Guardrails** — catch catastrophic mistakes only: blacklisted suppliers (token loss), premature close (quality), stuck loops (oscillation detection), invalid templates. Guardrails **block and redirect** — they retry the LLM with stronger context, never substitute the correct answer.

Key insight: the autopilot does NOT hardcode action parameters (suppliers, shipment IDs, templates).
That's the LLM's job. If the LLM makes a mistake, we improve the prompt, add entity context to
retry overrides, or use guardrails to block the bad action — never move the decision into the autopilot.

In [ ]:
import os
from google import genai

# os.environ['GEMINI_API_KEY'] = 'your-key-here'
try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except (ImportError, Exception):
    api_key = os.environ.get('GEMINI_API_KEY')

client = genai.Client(api_key=api_key)
print('Gemini client ready!')

In [ ]:
# ── Layer 1: Autopilot — ONLY queue management, alerts, lookups, boss fights ──
# The autopilot does NOT make decisions about actions, suppliers, or templates.
# Those are the LLM's job.

VALID_TEMPLATES = {
    'reorder_confirmation', 'delay_notification', 'quality_issue',
    'customs_update', 'rush_confirmation', 'launch_status',
    'compliance_status', 'dispute_resolution', 'general_update',
}


def _pick_next_order_llm(world):
    """Pick the next order, skipping those with unmet briefing prerequisites."""
    for candidate in world.queue:
        if candidate.status == OrderStatus.RESOLVED:
            continue
        if candidate.requires_briefing and not candidate.briefing_prepared:
            ready, _ = world.check_briefing_ready(candidate.id)
            if not ready:
                continue
        return candidate
    return world.queue[0] if world.queue else None


def _switch_away_from(order, world, skip_ids=None):
    """Find another order to work on when the current one is blocked.

    skip_ids: set of order IDs to also skip (e.g. orders the LLM is stuck on).
    """
    skip = skip_ids or set()
    for candidate in world.queue:
        if candidate.id == order.id or candidate.id in skip:
            continue
        if candidate.status == OrderStatus.RESOLVED:
            continue
        if candidate.requires_briefing and not candidate.briefing_prepared:
            r, _ = world.check_briefing_ready(candidate.id)
            if not r:
                continue
        return f'ACTION: read_order(order_id="{candidate.id}")'
    return 'ACTION: check_orders()'


def _auto_handle(agent, world, order):
    """Handle ONLY mechanical tasks. Everything else → LLM."""

    # No active order → pick next
    if order is None or order.status == OrderStatus.RESOLVED:
        next_order = _pick_next_order_llm(world)
        if next_order:
            return f'ACTION: read_order(order_id="{next_order.id}")'
        if world.queue:
            return f'ACTION: read_order(order_id="{world.queue[0].id}")'
        return 'ACTION: check_orders()'

    oid = order.id
    cat = order.category

    # System alerts: trivial, no decision needed
    if cat == OrderCategory.SYSTEM_ALERT:
        if not order.action_applied:
            return f'ACTION: acknowledge_alert(alert_id="{oid}")'
        next_order = _pick_next_order_llm(world)
        if next_order:
            return f'ACTION: read_order(order_id="{next_order.id}")'
        return 'ACTION: check_orders()'

    # KB lookup: always lookup first if required and not done
    if order.requires_lookup and not order.lookup_done:
        query = order.lookup_query or order.category.value
        return f'ACTION: lookup_kb(query="{query}")'

    # Boss fights: dependency resolution (algorithmic planning, not LLM)
    if cat == OrderCategory.LAUNCH:
        ready, _ = world.check_briefing_ready(oid)
        if ready and not order.briefing_prepared:
            return 'ACTION: prepare_launch_briefing()'
        if order.briefing_prepared and not order.action_applied:
            return f'ACTION: authorize_launch(order_id="{oid}")'
        if order.action_applied and not order.notification_sent:
            return 'ACTION: notify_client(template="launch_status")'
        if order.action_applied and order.notification_sent:
            return f'ACTION: close_order(order_id="{oid}")'
        world.active_order = None
        return _switch_away_from(order, world)

    if cat == OrderCategory.REGULATORY:
        ready, _ = world.check_briefing_ready(oid)
        if ready and not order.briefing_prepared:
            return 'ACTION: prepare_compliance_package()'
        if order.briefing_prepared and not order.action_applied:
            return f'ACTION: submit_compliance(order_id="{oid}")'
        if order.action_applied and not order.notification_sent:
            return 'ACTION: notify_client(template="compliance_status")'
        if order.action_applied and order.notification_sent:
            return f'ACTION: close_order(order_id="{oid}")'
        world.active_order = None
        return _switch_away_from(order, world)

    # Everything else → LLM decides
    return None


# ── Layer 2: Memory — structured context for the LLM ─────────────────────

def _extract_order_params(order):
    """Extract key parameters from order text so the LLM has them readily available."""
    import re
    params = {}

    # Extract shipment ID
    sh_match = re.search(r'#(SH-\d+)', order.message)
    if sh_match:
        params['shipment_id'] = sh_match.group(1)

    # Extract supplier name from entity
    supplier_patterns = [
        r'from\s+([\w\s]+(?:AG|GmbH|Ltd|Co|Shenzhen|Basel|Munich|Engineering))',
        r'Supplier:\s+([\w\s]+(?:AG|GmbH|Ltd|Co|Shenzhen|Basel|Munich|Engineering))',
        r'with\s+([\w\s]+(?:AG|GmbH|Ltd|Co|Shenzhen|Basel|Munich|Engineering))',
        r'confirm.*?spec.*?(MechPro\s+Engineering)',
    ]
    for pat in supplier_patterns:
        m = re.search(pat, order.message, re.IGNORECASE)
        if m:
            params['supplier'] = m.group(1).strip()
            break

    # Extract SKU
    sku_match = re.search(r'(SKU-\d+)', order.message)
    if sku_match:
        params['sku'] = sku_match.group(1)

    # Extract quantity
    qty_match = re.search(r'(\d+)\s+units', order.message)
    if qty_match:
        params['units_affected'] = qty_match.group(1)

    # Extract order value
    val_match = re.search(r'EUR\s+([\d,]+)', order.message)
    if val_match:
        params['value_eur'] = val_match.group(1).replace(',', '')

    # Extract batch number
    batch_match = re.search(r'batch\s+([\w-]+)', order.message, re.IGNORECASE)
    if batch_match:
        params['batch'] = batch_match.group(1)

    return params


def _build_memory(agent, world, order, history, turns_used):
    """Build context so the LLM can make informed decisions."""
    sections = []

    if order:
        entity = world.get_entity(order.entity_id)
        ename = entity.name if entity else 'Unknown'
        etier = entity.tier.value if entity else 'unknown'

        # Build detailed progress tracking
        progress = []
        if order.lookup_done:
            progress.append('KB lookup: DONE')
        if order.action_applied:
            progress.append('Main action: COMPLETED')
        else:
            progress.append('Main action: NOT DONE')
        if order.claim_filed:
            progress.append('Claim filed: YES')
        elif order.requires_claim:
            progress.append('Claim filed: NO (REQUIRED)')
        if order.escalated_to:
            progress.append(f'Escalated to: {order.escalated_to}')
        if order.notification_sent:
            progress.append('Client notified: YES')
        else:
            progress.append('Client notified: NO')

        # Extract key parameters from order text for easy reference
        params = _extract_order_params(order)
        entity_parts = []
        if 'supplier' in params:
            entity_parts.append(f'Supplier={params["supplier"]}')
        if 'shipment_id' in params:
            entity_parts.append(f'Shipment={params["shipment_id"]}')
        if 'sku' in params:
            entity_parts.append(f'SKU={params["sku"]}')
        if 'units_affected' in params:
            entity_parts.append(f'Units={params["units_affected"]}')
        if 'value_eur' in params:
            entity_parts.append(f'Value=EUR {params["value_eur"]}')
        if 'batch' in params:
            entity_parts.append(f'Batch={params["batch"]}')
        if entity_parts:
            progress.append(f'KEY ENTITIES: {" | ".join(entity_parts)}')

        # Show required next action type (not the full command — LLM decides params)
        if not order.action_applied and order.requires_action:
            progress.append(f'>>> REQUIRED NEXT ACTION TYPE: {order.requires_action}')
            progress.append('Extract parameters from KEY ENTITIES and order details above.')

        # For quality orders, show the multi-step checklist explicitly
        checklist = ''
        if order.category == OrderCategory.QUALITY:
            supplier_hint = params.get('supplier', 'see order details')
            steps = []
            steps.append(f'  Step 1 reject_shipment: {"DONE" if order.action_applied else "NOT DONE"}')
            steps.append(f'  Step 2 file_claim(supplier="{supplier_hint}", amount="estimated"): {"DONE" if order.claim_filed else "NOT DONE — DO THIS NEXT" if order.action_applied else "waiting"}')
            recent_actions = [h['content'] for h in history if h['role'] == 'action']
            reorder_done = any('place_order' in a for a in recent_actions[-15:])
            steps.append(f'  Step 3 place_order (alt supplier from KB): {"DONE" if reorder_done else "NOT DONE" if order.claim_filed else "waiting"}')
            steps.append(f'  Step 4 notify_client(template="quality_issue"): {"DONE" if order.notification_sent else "NOT DONE"}')
            steps.append(f'  Step 5 close_order: waiting')
            checklist = 'QUALITY CHECKLIST:\n' + '\n'.join(steps)

        progress_str = ' | '.join(p for p in progress if not p.startswith('>>>') and '\n' not in p)
        action_hint = '\n'.join(p for p in progress if p.startswith('>>>'))

        sections.append(
            f'=== ACTIVE ORDER ===\n'
            f'  {order.id}: {order.subject}\n'
            f'  Entity: {ename} ({etier}) | Priority: {order.priority.value}\n'
            f'  Category: {order.category.value}\n'
            f'  Progress: {progress_str}\n'
            + (f'\n{checklist}\n' if checklist else '')
            + (f'\n{action_hint}\n' if action_hint else '')
            + f'  Message: {order.message}'
        )

    # Include the last KB lookup result (full text — the LLM needs this to decide)
    for h in reversed(history):
        if h['role'] == 'result' and 'Knowledge Base' in h.get('content', ''):
            sections.append(f'=== KB PROCEDURE ===\n{h["content"]}')
            break

    turns_left = 100 - turns_used
    sections.append(
        f'=== PERFORMANCE ===\n'
        f'  Fulfilled: {agent.resolved_count}/{agent.WIN_RESOLVED} | '
        f'Quality: {agent.quality_score:.0f}% | '
        f'Tokens: {agent.operations_tokens}/{agent.MAX_TOKENS} | '
        f'Turns left: {turns_left}'
    )

    # Show recent actions so the LLM knows what it already did
    recent = [h for h in history[-10:] if h['role'] in ('action', 'result')]
    if recent:
        action_lines = []
        for h in recent[-8:]:
            prefix = 'You did' if h['role'] == 'action' else 'Result'
            action_lines.append(f'  {prefix}: {h["content"][:200]}')
        sections.append('=== RECENT ACTIONS ===\n' + '\n'.join(action_lines))

    return '\n\n'.join(sections)


# ── Layer 3: System prompt ────────────────────────────────────────────────

SYSTEM_PROMPT = """You are an AI operations coordinator processing supply chain orders.

GOAL: Fulfill orders efficiently with high quality. You have 3 operations tokens —
losing all means game over.

AVAILABLE TOOLS:
{tools}

CRITICAL RULE — ACT IMMEDIATELY AFTER KB LOOKUP:
After the KB lookup is done (shown as "KB lookup: DONE" in progress), your VERY NEXT
output MUST be an ACTION tool call — NOT read_order. Calling read_order after KB lookup
is a WASTE OF A TURN. You already have all the information you need.

Look at:
1. REQUIRED NEXT ACTION TYPE — tells you which tool to call
2. KEY ENTITIES — tells you the parameters (supplier, SKU, quantity, shipment ID)
3. KB PROCEDURE — tells you the procedure steps
4. Order message — has the full details

Then construct the action call with the right parameters.

PARAMETER EXTRACTION RULES:
- Supplier name: look at KEY ENTITIES "Supplier=..." or the entity name in the order
- SKU: look at KEY ENTITIES "SKU=..." or extract from order message (e.g., SKU-7712)
- Quantity: use KEY ENTITIES "Units=..." or a reasonable amount (reorder point minus current stock)
- Shipment ID: look at KEY ENTITIES "Shipment=..." (e.g., SH-1190)
- For MechPro spec orders: supplier="MechPro Engineering", sku="spec_v2.1", quantity="1"

TEMPLATE MATCHING (by order category):
  reorder → reorder_confirmation
  delay → delay_notification
  quality → quality_issue
  customs → customs_update
  rush → rush_confirmation
  dispute → dispute_resolution

SUPPLIER SELECTION:
- If the order names a specific supplier, use that supplier.
- For rush/reorder orders that do NOT name a supplier, pick an approved one:
    Parts/gears/bearings/valves/modules/electronics → SteelWorks GmbH
    Coatings/chemicals → PureChem Basel or CoatTech Munich
    Engineering specs → MechPro Engineering
- NEVER use QuickShip Ltd or BargainParts Co (blacklisted — costs a token!).

RUSH ORDER PROCEDURE:
  Step 1: KB lookup (automatic)
  Step 2: place_order(supplier="<approved>", sku="<from order>", quantity="<from order>", expedited="true")
          Orders under 1000 units do NOT need escalation. Just use expedited="true".
  Step 3: notify_client(template="rush_confirmation")
  Step 4: close_order(order_id="...")

REORDER PROCEDURE:
  Step 1: KB lookup (automatic)
  Step 2: place_order(supplier="<from order or approved>", sku="<from order>", quantity="<appropriate>")
  Step 3: notify_client(template="reorder_confirmation")
  Step 4: close_order(order_id="...")

DELAY PROCEDURE:
  Step 1: KB lookup (automatic)
  Step 2: update_timeline(order_id="<current order>", new_eta="<reasonable estimate>")
  Step 3: notify_client(template="delay_notification")
  Step 4: close_order(order_id="...")

QUALITY ORDER MULTI-STEP PROCEDURE:
  Step 1: reject_shipment(shipment_id="...", reason="...") — use the shipment ID from the order
  Step 2: file_claim(supplier="...", amount="...") — the supplier named in the order
  Step 3: place_order(supplier="...", sku="...", quantity="...") — use an ALTERNATIVE supplier
          from KB (e.g. PureChem Basel, CoatTech Munich). NOT the original supplier.
  Step 4: notify_client(template="quality_issue")
  Step 5: close_order(order_id="...")

CUSTOMS PROCEDURE:
  Step 1: KB lookup (automatic)
  Step 2: submit_documents(shipment_id="...", doc_type="certificate_of_origin")
  Step 3: notify_client(template="customs_update")
  Step 4: close_order(order_id="...")

DISPUTE ORDER PROCEDURE:
  Step 1: lookup_contract and verify_pricing first
  Step 2: If discrepancy > 5%: escalate(department="procurement", reason="pricing_discrepancy")
  Step 3: notify_client(template="dispute_resolution")
  Step 4: close_order(order_id="...")

RULES:
- NEVER call read_order when KB lookup is DONE and Main action is NOT DONE. Take the action.
- Extract parameters from KEY ENTITIES and the order message.
- Follow KB procedures step by step.
- Order value > EUR 50,000 → escalate to finance.
- When all steps are done: close the order immediately.

OUTPUT FORMAT:

REASONING: [One sentence: what step to do next]
ACTION: tool_name(param1="value1", param2="value2")
"""


# ── Layer 4: Guardrail helpers ────────────────────────────────────────────

def _retry_llm(user_msg, system, override_msg):
    """Re-invoke the LLM with a stronger prompt override."""
    try:
        retry = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=user_msg + override_msg,
            config=genai.types.GenerateContentConfig(
                system_instruction=system,
                max_output_tokens=300,
                temperature=0.4,
            ),
        )
        retry_text = retry.text
        retry_tool, retry_args = parse_action(retry_text)
        return retry_text, retry_tool, retry_args
    except Exception:
        return None


def _count_rereads_for_order(order, history):
    """Count how many times the LLM has re-read this order in recent history."""
    count = 0
    recent_actions = [h['content'] for h in history if h['role'] == 'action']
    for a in recent_actions[-30:]:
        if 'read_order' in a and order.id in a:
            count += 1
    return count


def _get_stuck_order_ids(history, threshold=3):
    """Find order IDs that have been re-read many times (stuck orders)."""
    import re as _re
    read_counts = {}
    recent_actions = [h['content'] for h in history if h['role'] == 'action']
    for a in recent_actions[-40:]:
        m = _re.search(r'read_order\(order_id="(T-\d+)"\)', a)
        if m:
            oid = m.group(1)
            read_counts[oid] = read_counts.get(oid, 0) + 1
    return {oid for oid, count in read_counts.items() if count >= threshold}


def _count_consecutive_reads(history):
    """Count consecutive read_order calls without any other action."""
    count = 0
    recent_actions = [h['content'] for h in history if h['role'] == 'action']
    for a in reversed(recent_actions):
        if 'read_order' in a:
            count += 1
        else:
            break
    return count


def _build_entity_override(order, params):
    """Build a context override string with extracted entities for retries."""
    parts = []
    parts.append(f"Order {order.id} ({order.category.value})")
    if 'supplier' in params:
        parts.append(f"Supplier: {params['supplier']}")
    if 'shipment_id' in params:
        parts.append(f"Shipment: {params['shipment_id']}")
    if 'sku' in params:
        parts.append(f"SKU: {params['sku']}")
    if 'units_affected' in params:
        parts.append(f"Quantity: {params['units_affected']}")
    if 'batch' in params:
        parts.append(f"Batch: {params['batch']}")

    # Add entity name from world context
    entity_name = order.subject.split('—')[-1].strip() if '—' in order.subject else ''
    if entity_name and 'supplier' not in params:
        parts.append(f"Entity: {entity_name}")

    return " | ".join(parts)


# ── Layer 5: The think function ───────────────────────────────────────────

def think_llm(agent: AgentState, world: SupplyWorld, history: list[dict]) -> str:
    """LLM-powered coordinator: autopilot for trivial/boss, LLM for decisions, guardrails for safety."""
    order = world.active_order
    turns_used = len([h for h in history if h['role'] == 'action'])

    # ── Autopilot: queue management, alerts, lookups, boss fights only ──
    auto = _auto_handle(agent, world, order)
    if auto:
        return auto

    # ── LLM: make the actual decision ──
    memory = _build_memory(agent, world, order, history, turns_used)
    system = SYSTEM_PROMPT.format(tools=TOOLS_DESCRIPTION)

    user_msg = f"""{memory}

Based on the order details and KB procedure, what is the next step?
If you've completed all required actions, notify the client then close."""

    # Detect stuck loops: count total re-reads for this order
    total_rereads = 0
    if order:
        total_rereads = _count_rereads_for_order(order, history)
        if total_rereads >= 1:
            params = _extract_order_params(order)
            entity_info = _build_entity_override(order, params)
            user_msg += (
                f"\n\nURGENT: You have re-read this order {total_rereads} time(s). "
                "STOP re-reading. Look at the PROGRESS, KEY ENTITIES, and REQUIRED NEXT ACTION TYPE above. "
                f"Context: {entity_info}. "
                "Output an ACTION call with the correct parameters NOW. Do NOT call read_order."
            )

    # Detect oscillation: consecutive read_order calls without any other action
    consec_reads = _count_consecutive_reads(history)
    if consec_reads >= 2:
        user_msg += (
            f"\n\nCRITICAL: You have called read_order {consec_reads} times in a row "
            "without taking any action. You are WASTING TURNS. "
            "STOP reading orders. Look at the ACTIVE ORDER and KEY ENTITIES above. "
            "Extract the parameters and take the required action NOW."
        )

    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=user_msg,
            config=genai.types.GenerateContentConfig(
                system_instruction=system,
                max_output_tokens=500,
                temperature=0.2,
            ),
        )
        text = response.text
    except Exception:
        # API error fallback: try to close gracefully
        if order and order.notification_sent:
            return f'ACTION: close_order(order_id="{order.id}")'
        if order:
            return 'ACTION: notify_client(template="general_update")'
        return 'ACTION: check_orders()'

    # ── Guardrails: catch catastrophic mistakes only ──
    tool_name, args = parse_action(text)

    # Guard: block blacklisted suppliers (costs a token!)
    if tool_name == 'place_order':
        supplier = args.get('supplier', '').lower()
        if any(b in supplier for b in ['quickship', 'bargainparts']):
            return 'ACTION: check_alternatives()'

    # Collect stuck order IDs for _switch_away_from
    stuck_ids = _get_stuck_order_ids(history, threshold=4) if order else set()

    # Guard: block read_order when we should be acting
    if tool_name == 'read_order' and order:
        target_id = args.get('order_id', '')
        params = _extract_order_params(order)
        entity_info = _build_entity_override(order, params)

        # Case 1: Re-reading the SAME order we already have open
        if target_id == order.id or target_id == '':
            if total_rereads < 2:
                # First retry: strong context with entities
                override = (
                    f"\n\nCRITICAL OVERRIDE: You ALREADY have order {order.id} open. "
                    f"DO NOT call read_order. This is a {order.category.value} order.\n"
                    f"Extracted context: {entity_info}\n"
                    f"Check the REQUIRED NEXT ACTION TYPE and KEY ENTITIES in the context above. "
                    f"What is the NEXT incomplete step? Output the ACTION call with parameters from the entities."
                )
                result = _retry_llm(user_msg, system, override)
                if result and result[1] != 'read_order':
                    return result[0]

            if total_rereads < 4:
                # Second tier: even more explicit retry
                action_type = order.requires_action or 'unknown'
                override = (
                    f"\n\nFINAL WARNING: You are stuck re-reading order {order.id}. "
                    f"This order needs: {action_type}\n"
                    f"Context: {entity_info}\n"
                    f"Use these entities to construct the {action_type}() call. "
                    f"Output ONLY: REASONING: ... ACTION: {action_type}(...)"
                )
                result = _retry_llm(user_msg, system, override)
                if result and result[1] not in ('read_order', 'check_orders'):
                    return result[0]
                # If still failing, try to advance: notify if action done, else switch
                if order.action_applied and not order.notification_sent:
                    return 'ACTION: notify_client(template="general_update")'
                if order.notification_sent:
                    return f'ACTION: close_order(order_id="{order.id}")'

            # Stuck too long — switch away to a non-stuck order
            world.active_order = None
            return _switch_away_from(order, world, skip_ids=stuck_ids)

        # Case 2: Switching to a DIFFERENT order when current order needs action
        if order.lookup_done and not order.action_applied:
            action_type = order.requires_action or 'unknown'
            override = (
                f"\n\nCRITICAL: You are trying to switch to a different order, but "
                f"order {order.id} is NOT DONE. You must complete it first.\n"
                f"This is a {order.category.value} order needing: {action_type}\n"
                f"Context: {entity_info}\n"
                f"Look at the KB PROCEDURE and construct the {action_type}() call "
                f"using the parameters from the context above."
            )
            result = _retry_llm(user_msg, system, override)
            if result and result[1] not in ('read_order', 'check_orders'):
                return result[0]

            # Retry with even stronger hint
            override2 = (
                f"\n\nYou MUST act on order {order.id} NOW. "
                f"Required action: {action_type}. "
                f"Context: {entity_info}. "
                f"Output: REASONING: Taking action. ACTION: {action_type}(...with params from context...)"
            )
            result2 = _retry_llm(user_msg, system, override2)
            if result2 and result2[1] not in ('read_order', 'check_orders'):
                return result2[0]

            # Both retries failed — if stuck too long, switch away
            if total_rereads >= 3:
                world.active_order = None
                return _switch_away_from(order, world, skip_ids=stuck_ids)

            # Not stuck yet, let the read go through (changes active order)
            # but only if oscillation isn't too bad
            if consec_reads >= 3:
                world.active_order = None
                return _switch_away_from(order, world, skip_ids=stuck_ids)

        # Case 3: Oscillation guard — consecutive reads without progress
        if consec_reads >= 3:
            action_type = order.requires_action or 'unknown'
            override = (
                f"\n\nCRITICAL: You are stuck in a read loop. "
                f"Order {order.id} needs: {action_type}\n"
                f"Context: {entity_info}\n"
                f"Output the ACTION call NOW."
            )
            result = _retry_llm(user_msg, system, override)
            if result and result[1] not in ('read_order', 'check_orders'):
                return result[0]

            # Switch away from stuck order
            world.active_order = None
            return _switch_away_from(order, world, skip_ids=stuck_ids)

    # Guard: block stuck loops (LLM outputting check_orders/check_stats with active order)
    if tool_name in ('check_orders', 'check_stats') and order:
        if order.notification_sent:
            return f'ACTION: close_order(order_id="{order.id}")'
        if consec_reads >= 2:
            world.active_order = None
            return _switch_away_from(order, world, skip_ids=stuck_ids)
        if order.action_applied and not order.notification_sent:
            return 'ACTION: notify_client(template="general_update")'
        return f'ACTION: read_order(order_id="{order.id}")'

    # Guard: don't close without required escalation — redirect to KB, don't hardcode
    if tool_name == 'close_order' and order:
        if order.requires_escalation and order.escalated_to != order.requires_escalation:
            override = (
                "\n\nCRITICAL: This order requires escalation before closing. "
                "Re-read the KB procedure — it specifies which department to escalate to "
                "and under what conditions. Check the dispute procedure or budget authority rules. "
                "Output: ACTION: escalate(department=\"...\", reason=\"...\")"
            )
            result = _retry_llm(user_msg, system, override)
            if result and result[1] == 'escalate':
                return result[0]
            # Second retry with more context
            override2 = (
                "\n\nYou MUST escalate this order before closing. "
                "The KB says pricing discrepancies > 5% must go to procurement. "
                "Orders > EUR 50,000 must go to finance. "
                "Which applies to this order? Escalate to the correct department NOW."
            )
            result2 = _retry_llm(user_msg, system, override2)
            if result2 and result2[1] == 'escalate':
                return result2[0]
            # Last resort: let the close go through (will lose points but not tokens)

    # Guard: don't close without required action applied
    if tool_name == 'close_order' and order:
        if order.requires_action and not order.action_applied:
            params = _extract_order_params(order)
            entity_info = _build_entity_override(order, params)
            action_type = order.requires_action
            override = (
                f"\n\nCRITICAL: You cannot close this order yet — the required action "
                f"'{action_type}' has not been applied.\n"
                f"Context: {entity_info}\n"
                f"Construct the {action_type}() call using these parameters."
            )
            result = _retry_llm(user_msg, system, override)
            if result and result[1] not in ('read_order', 'close_order', 'check_orders'):
                return result[0]

    # Guard: don't close without notifying — let the LLM pick the template
    if tool_name == 'close_order' and order:
        if not order.notification_sent:
            override = (
                "\n\nCRITICAL: You must notify the client BEFORE closing this order. "
                "Choose the appropriate template based on the order category:\n"
                "  reorder → reorder_confirmation, delay → delay_notification, "
                "quality → quality_issue, customs → customs_update, "
                "rush → rush_confirmation, dispute → dispute_resolution\n"
                "Output: ACTION: notify_client(template=\"...\")"
            )
            result = _retry_llm(user_msg, system, override)
            if result and result[1] == 'notify_client':
                template = result[2].get('template', '')
                if template in VALID_TEMPLATES:
                    return result[0]
            return 'ACTION: notify_client(template="general_update")'

    # Guard: invalid template name
    if tool_name == 'notify_client':
        template = args.get('template', '')
        if template not in VALID_TEMPLATES:
            return 'ACTION: notify_client(template="general_update")'

    # Guard: block boss fight shortcuts
    if tool_name == 'escalate' and order:
        if order.requires_briefing and not order.briefing_prepared:
            world.active_order = None
            return _switch_away_from(order, world)

    return text

In [ ]:
result = play_with_llm(
    think_fn=lambda agent, world, history: think_llm(agent, world, history),
    max_turns=100,
    delay=0.5,
)
print(f"\nResult: {'Victory!' if result['won'] else 'Not quite...'} | "
      f"Resolved: {result['resolved']} | Quality: {result['quality']}% | "
      f"Tokens: {result['tokens']} | Turns: {result['turns']}")
print(f"Log: {result['log_file']}")

In [ ]:
# Download the game log
try:
    from google.colab import files
    files.download(result['log_file'])
except ImportError:
    print(f"Log file: {result['log_file']}")
    print('Open the file to inspect your agent\'s decisions turn by turn.')